In [ ]:
import os
from pathlib import Path


def install(name):
    import subprocess
    subprocess.call(['pip', 'install',name])

# install('caas_jupyter_tools')
import mne
import pandas as pd
import pybv
# import caas_jupyter_tools as tools
import numpy as np

#### set params

In [6]:
csv_path = "./music_presentation_orders.csv"
csv_mapping = pd.read_csv(csv_path)
display(csv_mapping.head())

participant_id = 1

participant_trials = csv_mapping[csv_mapping['Participant_ID'] == participant_id].reset_index(drop=True)
participant_trials[['Participant_ID','Presentation_Order', 'Song_File', 'Original_Participant', 'Duration_Seconds']]


,Participant_ID,Presentation_Order,Song_File,Original_Participant,File_Path,Duration_Seconds,Randomization_Seed
0,1,1,4-1_proc.wav,4,music_stim/preprocesed/4/4-1_proc.wav,198.135875,2959848976
1,1,2,3-3_proc.wav,3,music_stim/preprocesed/3/3-3_proc.wav,253.283271,2959848976
2,1,3,5-3_proc.wav,5,music_stim/preprocesed/5/5-3_proc.wav,232.338875,2959848976
3,1,4,1-2_proc.wav,1,music_stim/preprocesed/1/1-2_proc.wav,252.424146,2959848976
4,1,5,4-2_proc.wav,4,music_stim/preprocesed/4/4-2_proc.wav,301.069937,2959848976


,Participant_ID,Presentation_Order,Song_File,Original_Participant,Duration_Seconds
0,1,1,4-1_proc.wav,4,198.135875
1,1,2,3-3_proc.wav,3,253.283271
2,1,3,5-3_proc.wav,5,232.338875
3,1,4,1-2_proc.wav,1,252.424146
4,1,5,4-2_proc.wav,4,301.069937
5,1,6,5-2_proc.wav,5,228.948771
6,1,7,2-1_proc.wav,2,324.764458
7,1,8,2-2_proc.wav,2,397.316646
8,1,9,5-1_proc.wav,5,237.075750
9,1,10,2-3_proc.wav,2,363.472125


In [7]:

def get_s1_code(event_id_dict):
    for k, v in event_id_dict.items():
        ks = k.strip().replace("  ", " ")
        if ks == "Stimulus/S 1" or k.strip() == "Stimulus/S  1":
            return v
    raise KeyError(f"Could not find 'Stimulus/S  1' in event_id: {list(event_id_dict.keys())}")


### loop through participant data

In [ ]:
# num=5
vhdr_data_root="./output/"
vhdr_participant_id_prefix="pilot_"

for id in range(5,6):
    print(id)

    participant_id = id
    full_participant_id=vhdr_participant_id_prefix+str(participant_id)

    # participant_trials = csv_mapping[csv_mapping['Participant_ID'] == participant_id].reset_index(drop=True)
    # participant_trials[['Participant_ID','Presentation_Order', 'Song_File', 'Original_Participant', 'Duration_Seconds']]

    vhdr_path = vhdr_data_root+full_participant_id+"/"+full_participant_id+"_cortical_preprocessed.vhdr"
    output_dir = Path(vhdr_data_root+full_participant_id+"/preprocessed_trials")
    # +pilot_"+str(participant_id))
    print(vhdr_path,output_dir)
    output_dir.mkdir(exist_ok=True)
    
    raw_data = mne.io.read_raw_brainvision(vhdr_path, preload=False)
    events, event_id = mne.events_from_annotations(raw_data) #get events & event ID

    s1_code = get_s1_code(event_id)
    s1_onsets = events[:, 0]
    s1_indices = np.where(events[:, 2] == s1_code)[0]

    # Extract s1events 
    s1_samples = s1_onsets[s1_indices]
    print(s1_indices)
    needed = 5 + len(participant_trials)
    skip_range = (51260000, 56280000)

    if participant_id==1:
        participant_trials = participant_trials.drop(participant_trials[participant_trials["Song_File"] == "2-1_proc.wav"].index).reset_index(drop=True)

        valid_indices = []
        needed=-1
        for i, s1_sample in zip(s1_indices, s1_samples):
            if skip_range[0] < s1_sample < skip_range[1]:
                print(f"Skipping trial with S1 at sample {s1_sample} (faulty run)")
                continue
            valid_indices.append(i)

        # Update s1_indices to exclude the bad one
        s1_indices = np.array(valid_indices)

    # Final check
    if len(s1_indices) < needed:
        raise ValueError(f"Only found {len(s1_indices)} valid S1 starts, but expected {needed}.")

    click_trial_count = 5  # first 5 are ABR licks
    trial_metadata = []

    # sampling freq
    sfreq = raw_data.info['sfreq']
    print("sampling frequency",sfreq)

    # ---- CLICK TRIALS ---
    for i in range(5):
        start_index = s1_indices[i]
        start_sample = events[start_index, 0]
        start_time = start_sample / sfreq
        duration_sec = 60.0  # ixed

        tmin = start_time
        tmax = start_time + duration_sec

        trial_raw = raw_data.copy().crop(tmin=tmin, tmax=tmax)
        trial_events, trial_event_id = mne.events_from_annotations(trial_raw)

        presentation_order = i + 1
        song_base = "click"
        orig_ptp = -1  
        new_base_name_click = f"pilot_{participant_id}_{song_base}_trial{presentation_order}_cortical_preproc"
        print(f"[CLICK] Saving: {new_base_name_click}")

        fif_path = output_dir / f"{new_base_name_click}.fif"
        trial_raw.save(fif_path, overwrite=True)

        brainvision_path = output_dir / new_base_name_click
        mne.export.export_raw(str(brainvision_path) + ".vhdr", trial_raw, overwrite=True)

        if len(trial_events) > 0:
            first_event = trial_events[0:1]
            code = first_event[0, 2]
            label = [k for k, v in trial_event_id.items() if v == code][0]
            epochs = mne.Epochs(trial_raw, first_event, event_id={label: code},
                                tmin=0, tmax=duration_sec, baseline=None, preload=True)
            epochs.save(output_dir / f"{new_base_name_click}_epo.fif", overwrite=True)
        else:
            print(f" No events in click trial {i+1}, skipping epoch.")

    output_dir.mkdir(exist_ok=True)
    for i, row in participant_trials.iterrows():
        # === =trial timing info ===
        start_index = s1_indices[5 + i]   # skip the 5 click-trial S1s
        start_sample = events[start_index, 0]
        start_time = start_sample / sfreq
        duration_sec = row["Duration_Seconds"]
        print(duration_sec)
        print(f"Trial {i+1}: Duration = {duration_sec:.2f}s")

        # === Song timing ===
        trial_type = "song"
        tmin = start_time
        # tmax = start_time + duration_sec + 1
        # tmax = min(start_time + duration_sec + 1, raw_data.times[-1])
        tmax = min(start_time + duration_sec + 1, raw_data.times[-1])

        if tmax <= tmin:
            raise ValueError(f"Invalid crop range for trial {i+1}: tmin={tmin:.2f}, tmax={tmax:.2f}")

        # === Crop raw ===
        trial_raw = raw_data.copy().crop(tmin=tmin, tmax=tmax)
        trial_events, trial_event_id = mne.events_from_annotations(trial_raw)

        # === File naming ===
        song_base = row["Song_File"].split(".")[0]
        orig_ptp = int(row["Original_Participant"])
        presentation_order = int(row["Presentation_Order"])
        new_base_name_song = f"pilot_{participant_id}-trial{presentation_order}_{song_base}_originalptp-{orig_ptp}_cortical_preproc"
        print(f"Saving: {new_base_name_song}")

        # === Save raw trial as .fif ===
        fif_path = output_dir / f"{new_base_name_song}.fif"
        trial_raw.save(fif_path, overwrite=True)

        # === Save as BrainVision (.vhdr, .eeg, .vmrk) ===
        brainvision_path = output_dir / new_base_name_song
        trial_raw.set_annotations(trial_raw.annotations.crop(0, trial_raw.times[-1]))

        mne.export.export_raw(str(brainvision_path) + ".vhdr", trial_raw, overwrite=True)

        # === Epoch ===
        is_click=False
        if len(trial_events) > 0:
            first_event = trial_events[0:1]
            event_code = first_event[0, 2]
            event_label = [k for k, v in trial_event_id.items() if v == event_code][0]

            epoch_tmin = 0
            epoch_tmax = (60 if is_click else duration_sec + 1)

            epochs = mne.Epochs(
                trial_raw,
                first_event,
                event_id={event_label: event_code},
                tmin=epoch_tmin,
                tmax=epoch_tmax,
                baseline=None,
                preload=True
            )

            epochs_path = output_dir / f"{new_base_name_song}_epo.fif"
            epochs.save(epochs_path, overwrite=True)
        else:
            print(f" No events in trial {i + 1}, skipping epoch.")

        # ===  metadata ===
        trial_metadata.append({
            "Trial_Number": i + 1,
            "Presentation_Order": presentation_order,
            "Original_Participant": orig_ptp,
            "Song_File": row["Song_File"],
            "Duration_Seconds": duration_sec,
            "Trial_Type": trial_type,
            "Base_Name": new_base_name_song,
            "Start_Sample": start_sample,
            "Tmin": tmin,
            "Tmax": tmax
        })

    # Save metadata CSV
    metadata_df = pd.DataFrame(trial_metadata)
    metadata_csv_path = output_dir / f"pilot_{participant_id}_trial_metadata.csv"
    metadata_df.to_csv(metadata_csv_path, index=False)

5
./output/pilot_5/pilot_5_cortical_preprocessed.vhdr output/pilot_5/preprocessed_trials
Extracting parameters from ./output/pilot_5/pilot_5_cortical_preprocessed.vhdr...
Setting channel info structure...
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[  0   7  14  21  28  35  42  49  56  63  70  77  84  91  98 105 111 118
 125 132]
sampling frequency 128.0
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4')]
[CLICK] Saving: pilot_5_click_trial1_cortical_preproc
Overwriting existing file.
Writing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial1_cortical_preproc.fif
Overwriting existing file.
Closing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial1_cortical_preproc.fif
[done]
Overwriting existing file.
Not setting metadata
1 matching events 

/tmp/ipykernel_86752/4125101315.py:78: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial1_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)


Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_5_click_trial2_cortical_preproc
Overwriting existing file.
Writing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial2_cortical_preproc.fif
Overwriting existing file.
Closing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial2_cortical_preproc.fif
[done]
Overwriting existing file.
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 7681 original time points ...
0 bad epochs dropped
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_5_click_trial3_cortical_preproc
Overw

/tmp/ipykernel_86752/4125101315.py:78: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial2_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)
/tmp/ipykernel_86752/4125101315.py:78: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial3_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)
/tmp/ipykernel_86752/4125101315.py:78: RuntimeWarning: This filename (/scr

Overwriting existing file.
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 7681 original time points ...
0 bad epochs dropped
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_5_click_trial5_cortical_preproc
Overwriting existing file.
Writing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial5_cortical_preproc.fif
Overwriting existing file.
Closing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5_click_trial5_cortical_preproc.fif
[done]
Overwriting existing file.
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 7681 original time points ...
0 b

...

/tmp/ipykernel_86752/4125101315.py:128: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial13_3-2_proc_originalptp-3_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)


379.25152083333336
Trial 14: Duration = 379.25s
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
Saving: pilot_5-trial14_1-1_proc_originalptp-1_cortical_preproc
Writing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial14_1-1_proc_originalptp-1_cortical_preproc.fif
Closing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial14_1-1_proc_originalptp-1_cortical_preproc.fif


/tmp/ipykernel_86752/4125101315.py:128: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial14_1-1_proc_originalptp-1_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)


[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 48673 original time points ...
0 bad epochs dropped
217.4084375
Trial 15: Duration = 217.41s
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
Saving: pilot_5-trial15_4-3_proc_originalptp-4_cortical_preproc
Writing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial15_4-3_proc_originalptp-4_cortical_preproc.fif
Closing /scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial15_4-3_proc_originalptp-4_cortical_preproc.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 27957 original time points ...
0 bad epochs dropped


/tmp/ipykernel_86752/4125101315.py:128: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/output/pilot_5/preprocessed_trials/pilot_5-trial15_4-3_proc_originalptp-4_cortical_preproc.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=True)


### sanity check

In [ ]:
output_dir = Path(output_dir) 

trial_files = sorted(list(output_dir.glob("*.fif")))
trial_files = [f for f in trial_files if not f.name.endswith("_epo.fif")]


for fif_file in trial_files:
    base_name = fif_file.stem.replace("_raw", "").replace("_epo", "")
    brainvision_base = output_dir / base_name

    print(f"\n🔎 Checking: {base_name}")

    # === Check .fif ===""
    try:
        raw = mne.io.read_raw_fif(fif_file, preload=False, verbose="ERROR")
        fif_duration = raw.times[-1] - raw.times[0]
        fif_events, _ = mne.events_from_annotations(raw, verbose="ERROR")
        print(f"  ✓ .fif duration: {fif_duration:.2f}s | Events: {len(fif_events)}")
    except Exception as e:
        print(f"Could not load .fif: {e}")
        continue

    # === Check BV ===
    vhdr_path = brainvision_base.with_suffix(".vhdr")
    vmrk_path = brainvision_base.with_suffix(".vmrk")
    eeg_path = brainvision_base.with_suffix(".eeg")

    if not (vhdr_path.exists() and vmrk_path.exists() and eeg_path.exists()):
        print("Missing BrainVision files")
        continue

    try:
        bv_raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
        bv_duration = bv_raw.times[-1] - bv_raw.times[0]
        bv_events, _ = mne.events_from_annotations(bv_raw, verbose="ERROR")
        print(f"  ✓ .vhdr/.eeg duration: {bv_duration:.2f}s | Events: {len(bv_events)}")
    except Exception as e:
        print(f"Could not load BrainVision files: {e}")
        continue

    if abs(fif_duration - bv_duration) > 0.01:
        print("Duration mismatch between .fif and .vhdr")

    if len(fif_events) != len(bv_events):
        print("Event count mismatch between .fif and .vmrk")

    # === Check first last marker sample
    marker_samples = bv_events[:, 0] if len(bv_events) > 0 else []
    if marker_samples.any():
        first_marker = marker_samples.min() / bv_raw.info["sfreq"]
        last_marker = marker_samples.max() / bv_raw.info["sfreq"]
        if first_marker < 0 or last_marker > bv_duration:
            print(f"Marker out of range: first={first_marker:.2f}s, last={last_marker:.2f}s")

print("\n Validation complete.")


🔎 Checking: pilot_1-trial10_2-3_proc_originalptp-2
  ✓ .fif duration: 364.47s | Events: 7
  ✓ .vhdr/.eeg duration: 364.47s | Events: 7

🔎 Checking: pilot_1-trial11_1-3_proc_originalptp-1
  ✓ .fif duration: 244.23s | Events: 7
  ✓ .vhdr/.eeg duration: 244.23s | Events: 7

🔎 Checking: pilot_1-trial12_3-1_proc_originalptp-3
  ✓ .fif duration: 230.13s | Events: 6
  ✓ .vhdr/.eeg duration: 230.13s | Events: 7
  ⚠️ Event count mismatch between .fif and .vmrk

🔎 Checking: pilot_1-trial13_3-2_proc_originalptp-3
  ✓ .fif duration: 228.09s | Events: 7
  ✓ .vhdr/.eeg duration: 228.09s | Events: 7

🔎 Checking: pilot_1-trial14_1-1_proc_originalptp-1
  ✓ .fif duration: 380.25s | Events: 7
  ✓ .vhdr/.eeg duration: 380.25s | Events: 7

🔎 Checking: pilot_1-trial15_4-3_proc_originalptp-4
  ✓ .fif duration: 218.41s | Events: 7
  ✓ .vhdr/.eeg duration: 218.41s | Events: 7

🔎 Checking: pilot_1-trial1_4-1_proc_originalptp-4
  ✓ .fif duration: 199.14s | Events: 7
  ✓ .vhdr/.eeg duration: 199.14s | Events: 7


# Single subject below
### === CONFIGURATION ===


In [20]:
participant_id = 4
csv_path = "./music_presentation_orders.csv"
vhdr_path = "pilot_"+str(participant_id)+".vhdr"
output_dir = Path("pilot_4_trials_output")
output_dir.mkdir(exist_ok=True)

### === load CSV ===


In [21]:
participant_trials = csv_mapping[csv_mapping['Participant_ID'] == participant_id].reset_index(drop=True)
participant_trials[['Participant_ID','Presentation_Order', 'Song_File', 'Original_Participant', 'Duration_Seconds']]

,Participant_ID,Presentation_Order,Song_File,Original_Participant,Duration_Seconds
0,4,1,2-2_proc.wav,2,397.316646
1,4,2,2-3_proc.wav,2,363.472125
2,4,3,1-2_proc.wav,1,252.424146
3,4,4,4-1_proc.wav,4,198.135875
4,4,5,4-3_proc.wav,4,217.408437
5,4,6,1-3_proc.wav,1,243.229042
6,4,7,4-2_proc.wav,4,301.069937
7,4,8,5-3_proc.wav,5,232.338875
8,4,9,3-2_proc.wav,3,227.091167
9,4,10,5-1_proc.wav,5,237.075750


### === load EEG ===


In [22]:
raw_data = mne.io.read_raw_brainvision(vhdr_path, preload=False)
events, event_id = mne.events_from_annotations(raw_data) #get events and event ID

def get_s1_code(event_id_dict):
    for k, v in event_id_dict.items():
        ks = k.strip().replace("  ", " ")
        if ks == "Stimulus/S 1" or k.strip() == "Stimulus/S  1":
            return v
    raise KeyError(f"Could not find 'Stimulus/S  1' in event_id: {list(event_id_dict.keys())}")

s1_code = get_s1_code(event_id)

import numpy as np
s1_indices = np.where(events[:, 2] == s1_code)[0]

needed = 5 + len(participant_trials)
if len(s1_indices) < needed:
    raise ValueError(f"Found {len(s1_indices)} S1 starts, need {needed}.")

click_trial_count = 5  # First 5 trials are click trials

# trial_start_indices = [1, 7, 14, 21, 28, 35, 42, 49, 56, 63, 65, 72, 79, 86, 93]  # Provided externally

trial_metadata = []


#get sampling freq
sfreq = raw_data.info['sfreq']
print("sampling frequency",sfreq)

Extracting parameters from pilot_4.vhdr...
Setting channel info structure...
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
sampling frequency 25000.0


### extract clicks

In [ ]:
for i in range(5):
    start_index = s1_indices[i]
    start_sample = events[start_index, 0]
    start_time = start_sample / sfreq
    duration_sec = 60.0  # fixed for clicks

    tmin = start_time
    tmax = start_time + duration_sec

    trial_raw = raw_data.copy().crop(tmin=tmin, tmax=tmax)
    trial_events, trial_event_id = mne.events_from_annotations(trial_raw)

    presentation_order = i + 1
    song_base = "click_trial"
    orig_ptp = -1  
    new_base_name = f"pilot_{participant_id}-trial{presentation_order}_{song_base}_originalptp-{orig_ptp}"
    print(f"[CLICK] Saving: {new_base_name}")

    fif_path = output_dir / f"{new_base_name}.fif"
    trial_raw.save(fif_path, overwrite=False)

    brainvision_path = output_dir / new_base_name
    mne.export.export_raw(str(brainvision_path) + ".vhdr", trial_raw, overwrite=False)

    if len(trial_events) > 0:
        first_event = trial_events[0:1]
        code = first_event[0, 2]
        label = [k for k, v in trial_event_id.items() if v == code][0]
        epochs = mne.Epochs(trial_raw, first_event, event_id={label: code},
                            tmin=0, tmax=duration_sec, baseline=None, preload=True)
        epochs.save(output_dir / f"{new_base_name}_epo.fif", overwrite=False)
    else:
        print(f"No events in click trial {i+1}, skipping epoch.")


Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4')]
[CLICK] Saving: pilot_4-trial1_click_trial_originalptp--1
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial1_click_trial_originalptp--1.fif


/tmp/ipykernel_179292/169486046.py:23: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial1_click_trial_originalptp--1.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=False)


Closing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial1_click_trial_originalptp--1.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 1500001 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_4-trial2_click_trial_originalptp--1
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial2_click_trial_originalptp--1.fif


/tmp/ipykernel_179292/169486046.py:23: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial2_click_trial_originalptp--1.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=False)


Closing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial2_click_trial_originalptp--1.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 1500001 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_4-trial3_click_trial_originalptp--1
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial3_click_trial_originalptp--1.fif


...

Closing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial3_click_trial_originalptp--1.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 1500001 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_4-trial4_click_trial_originalptp--1
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial4_click_trial_originalptp--1.fif


/tmp/ipykernel_179292/169486046.py:23: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial4_click_trial_originalptp--1.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=False)


Closing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial4_click_trial_originalptp--1.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 1500001 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
[CLICK] Saving: pilot_4-trial5_click_trial_originalptp--1
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial5_click_trial_originalptp--1.fif


/tmp/ipykernel_179292/169486046.py:23: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial5_click_trial_originalptp--1.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=False)


Closing /scratch/users/daelsaid/eeg_music_preference/pilot_4_trials_output/pilot_4-trial5_click_trial_originalptp--1.fif
[done]
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Loading data for 1 events and 1500001 original time points ...
0 bad epochs dropped


### extract songs 

In [ ]:
output_dir.mkdir(exist_ok=True)


for i, row in participant_trials.iterrows():
    # ===  trial timing info ===
    start_index = s1_indices[5 + i]   # skip the 5 click-trial S1s
    start_sample = events[start_index, 0]
    start_time = start_sample / sfreq
    duration_sec = row["Duration_Seconds"]
    print(duration_sec)
    print(f"Trial {i+1}: Duration = {duration_sec:.2f}s")

    # === sogn timing ===
    trial_type = "song"
    tmin = start_time
    tmax = start_time + duration_sec + 1

    # === corp the raw data ===
    trial_raw = raw_data.copy().crop(tmin=tmin, tmax=tmax)
    trial_events, trial_event_id = mne.events_from_annotations(trial_raw)

    # === File naming ===
    song_base = row["Song_File"].split(".")[0]
    orig_ptp = int(row["Original_Participant"])
    presentation_order = int(row["Presentation_Order"])
    new_base_name = f"pilot_{participant_id}-trial{presentation_order}_{song_base}_originalptp-{orig_ptp}"
    print(f"saving: {new_base_name}")


    # === Save raw trial as .fif ===
    fif_path = output_dir / f"{new_base_name}.fif"
    trial_raw.save(fif_path, overwrite=False)

    # === Save as BrainVision (.vhdr, .eeg, .vmrk) ===
    brainvision_path = output_dir / new_base_name
    mne.export.export_raw(str(brainvision_path) + ".vhdr", trial_raw, overwrite=False)

    # === Epoch ===
    if len(trial_events) > 0:
        first_event = trial_events[0:1]
        event_code = first_event[0, 2]
        event_label = [k for k, v in trial_event_id.items() if v == event_code][0]

        epoch_tmin = 0
        epoch_tmax = (60 if is_click else duration_sec + 1)

        epochs = mne.Epochs(
            trial_raw,
            first_event,
            event_id={event_label: event_code},
            tmin=epoch_tmin,
            tmax=epoch_tmax,
            baseline=None,
            preload=True
        )

        epochs_path = output_dir / f"{new_base_name}_epo.fif"
        epochs.save(epochs_path, overwrite=False)
    else:
        print(f"No events in trial {i + 1}, skipping epoch.")

    # === Save metadata ===
    trial_metadata.append({
        "Trial_Number": i + 1,
        "Presentation_Order": presentation_order,
        "Original_Participant": orig_ptp,
        "Song_File": row["Song_File"],
        "Duration_Seconds": duration_sec,
        "Trial_Type": trial_type,
        "Base_Name": new_base_name,
        "Start_Sample": start_sample,
        "Tmin": tmin,
        "Tmax": tmax
    })

# Save metadata CSV
metadata_df = pd.DataFrame(trial_metadata)
metadata_csv_path = output_dir / f"pilot_{participant_id}_trial_metadata.csv"
metadata_df.to_csv(metadata_csv_path, index=False)

# import caas_jupyter_tools as tools; tools.display_dataframe_to_user(name="Trial Metadata", dataframe=metadata_df)



198.135875
Trial 1: Duration = 198.14s
Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S  2'), np.str_('Stimulus/S  4'), np.str_('Stimulus/S  8')]
Saving: pilot_1-trial1_4-1_proc_originalptp-4
Writing /scratch/users/daelsaid/eeg_music_preference/pilot_1_trials_output/pilot_1-trial1_4-1_proc_originalptp-4.fif


/tmp/ipykernel_249502/2865385112.py:33: RuntimeWarning: This filename (/scratch/users/daelsaid/eeg_music_preference/pilot_1_trials_output/pilot_1-trial1_4-1_proc_originalptp-4.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  trial_raw.save(fif_path, overwrite=False)


Closing /scratch/users/daelsaid/eeg_music_preference/pilot_1_trials_output/pilot_1-trial1_4-1_proc_originalptp-4.fif
[done]


NameError: name 'is_click' is not defined

In [ ]:
participant_id = 4
csv_path = "./music_presentation_orders.csv"
vhdr_path = "pilot_"+str(participant_id)+".vhdr"
output_dir = Path("pilot_4_trials_output")
output_dir.mkdir(exist_ok=True)

participant_trials = csv_mapping[csv_mapping['Participant_ID'] == participant_id].reset_index(drop=True)
participant_trials[['Participant_ID','Presentation_Order', 'Song_File', 'Original_Participant', 'Duration_Seconds']]


# >>> INSERT THIS WHOLE BLOCK JUST BEFORE your existing song loop

# ---- CLICK TRIALS (first 5 S1s), fixed 60s ----
for i in range(5):
    start_index = s1_indices[i]
    start_sample = events[start_index, 0]
    start_time = start_sample / sfreq
    duration_sec = 60.0  # fixed for clicks

    tmin = start_time
    tmax = start_time + duration_sec

    trial_raw = raw_data.copy().crop(tmin=tmin, tmax=tmax)
    trial_events, trial_event_id = mne.events_from_annotations(trial_raw)

    presentation_order = i + 1
    song_base = "click_trial"
    orig_ptp = -1  # or 0 if you prefer
    new_base_name = f"pilot_{participant_id}-trial{presentation_order}_{song_base}_originalptp-{orig_ptp}"
    print(f"[CLICK] Saving: {new_base_name}")

    fif_path = output_dir / f"{new_base_name}.fif"
    trial_raw.save(fif_path, overwrite=False)

    brainvision_path = output_dir / new_base_name
    mne.export.export_raw(str(brainvision_path) + ".vhdr", trial_raw, overwrite=False)

    if len(trial_events) > 0:
        first_event = trial_events[0:1]
        code = first_event[0, 2]
        label = [k for k, v in trial_event_id.items() if v == code][0]
        epochs = mne.Epochs(trial_raw, first_event, event_id={label: code},
                            tmin=0, tmax=duration_sec, baseline=None, preload=True)
        epochs.save(output_dir / f"{new_base_name}_epo.fif", overwrite=False)
    else:
        print(f"⚠️ No events in click trial {i+1}, skipping epoch.")



### single subject sanity check

In [ ]:
display()
# output_dir = Path(output_dir) 

trial_files = sorted(list(output_dir.glob("*.fif")))
trial_files = [f for f in trial_files if not f.name.endswith("_epo.fif")]


for fif_file in trial_files:
    base_name = fif_file.stem.replace("_raw", "").replace("_epo", "")
    brainvision_base = output_dir / base_name

    print(f"\n Checking: {base_name}")

    # === Check .fif ===""
    try:
        raw = mne.io.read_raw_fif(fif_file, preload=False, verbose="ERROR")
        fif_duration = raw.times[-1] - raw.times[0]
        fif_events, _ = mne.events_from_annotations(raw, verbose="ERROR")
        print(f"  ✓ .fif duration: {fif_duration:.2f}s | Events: {len(fif_events)}")
    except Exception as e:
        print(f" Could not load .fif: {e}")
        continue

    # === Check BrainVision triplet ===
    vhdr_path = brainvision_base.with_suffix(".vhdr")
    vmrk_path = brainvision_base.with_suffix(".vmrk")
    eeg_path = brainvision_base.with_suffix(".eeg")

    if not (vhdr_path.exists() and vmrk_path.exists() and eeg_path.exists()):
        print(" Missing BrainVision files")
        continue

    try:
        bv_raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
        bv_duration = bv_raw.times[-1] - bv_raw.times[0]
        bv_events, _ = mne.events_from_annotations(bv_raw, verbose="ERROR")
        print(f"  ✓ .vhdr/.eeg duration: {bv_duration:.2f}s | Events: {len(bv_events)}")
    except Exception as e:
        print(f" Could not load BrainVision files: {e}")
        continue

    # === Sanity Check ===
    if abs(fif_duration - bv_duration) > 0.01:
        print("Duration mismatch between .fif and .vhdr")

    if len(fif_events) != len(bv_events):
        print("Event count mismatch between .fif and .vmrk")

    # === Check first and last marker sample
    marker_samples = bv_events[:, 0] if len(bv_events) > 0 else []
    if marker_samples.any():
        first_marker = marker_samples.min() / bv_raw.info["sfreq"]
        last_marker = marker_samples.max() / bv_raw.info["sfreq"]
        if first_marker < 0 or last_marker > bv_duration:
            print(f"Marker out of range: first={first_marker:.2f}s, last={last_marker:.2f}s")

print("\nValidation complete.")



🔎 Checking: pilot_1-trial10_2-3_proc_originalptp-2
  ✓ .fif duration: 364.47s | Events: 7
  ✓ .vhdr/.eeg duration: 364.47s | Events: 7

🔎 Checking: pilot_1-trial11_1-3_proc_originalptp-1
  ✓ .fif duration: 244.23s | Events: 7
  ✓ .vhdr/.eeg duration: 244.23s | Events: 7

🔎 Checking: pilot_1-trial12_3-1_proc_originalptp-3
  ✓ .fif duration: 230.13s | Events: 6
  ✓ .vhdr/.eeg duration: 230.13s | Events: 7
  ⚠️ Event count mismatch between .fif and .vmrk

🔎 Checking: pilot_1-trial13_3-2_proc_originalptp-3
  ✓ .fif duration: 228.09s | Events: 7
  ✓ .vhdr/.eeg duration: 228.09s | Events: 7

🔎 Checking: pilot_1-trial14_1-1_proc_originalptp-1
  ✓ .fif duration: 380.25s | Events: 7
  ✓ .vhdr/.eeg duration: 380.25s | Events: 7

🔎 Checking: pilot_1-trial15_4-3_proc_originalptp-4
  ✓ .fif duration: 218.41s | Events: 7
  ✓ .vhdr/.eeg duration: 218.41s | Events: 7

🔎 Checking: pilot_1-trial1_4-1_proc_originalptp-4
  ✓ .fif duration: 199.14s | Events: 7
  ✓ .vhdr/.eeg duration: 199.14s | Events: 7
